# 05 — Evaluación y comparación formal

Orden obligatorio para cada variante: calibrar solo con validation, congelar el threshold, evaluar test limpio y ejecutar stress tests por separado.

```bash
python -m src.evaluation.evaluate --mode calibrate --experiment-name baseline_formal/<variante> --batch-size 32 --criterion max_f1
python -m src.evaluation.evaluate --mode test --experiment-name baseline_formal/<variante> --batch-size 32
python -m src.evaluation.evaluate_stress --experiment-name baseline_formal/<variante> --batch-size 32 --seed 2026
python -m src.evaluation.compare_baselines
```

El criterio maximiza F1 en validation; si hay empate, prioriza menor FAR y luego menor FRR. Test nunca selecciona el threshold.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
FORMAL_DIR = PROJECT_ROOT / 'outputs/experiments/baseline_formal'
VARIANTS = {
    'Con aumento': FORMAL_DIR / 'baseline_con_aumento',
    'Sin aumento': FORMAL_DIR / 'baseline_sin_aumento',
}

In [ ]:
rows = []
for variant, directory in VARIANTS.items():
    for split, filename in [('validation', 'validation_metrics.json'), ('test_clean', 'test_metrics.json')]:
        metrics = json.loads((directory / filename).read_text(encoding='utf-8'))
        rows.append({'variant': variant, 'split': split, **{key: metrics[key] for key in ('threshold', 'accuracy', 'precision', 'recall', 'f1', 'far', 'frr', 'roc_auc')}})
display(pd.DataFrame(rows))

In [ ]:
stress = []
for variant, directory in VARIANTS.items():
    frame = pd.read_csv(directory / 'stress_metrics.csv')
    frame.insert(0, 'variant', variant)
    stress.append(frame)
stress_df = pd.concat(stress, ignore_index=True)
display(stress_df[['variant', 'condition', 'accuracy', 'f1', 'far', 'frr']])
display(stress_df.groupby('variant')[['accuracy', 'f1', 'far', 'frr']].mean())

In [ ]:
comparison_dir = FORMAL_DIR / 'comparison'
for filename in ('clean_test_comparison.png', 'stress_comparison.png'):
    path = comparison_dir / filename
    if path.exists(): display(Image(filename=str(path)))
for variant, directory in VARIANTS.items():
    print()
    print(variant)
    display(Image(filename=str(directory / 'validation_confusion_matrix.png')))
    display(Image(filename=str(directory / 'test_confusion_matrix.png')))

Las métricas provienen de splits por video auditados sin fuga. Los stress tests alteran solo `image_b` (probe), permanecen separados del test limpio y no cambian el threshold. Ejecutar las celdas en orden; no se guardan outputs pesados.